<a href="https://colab.research.google.com/github/charmy-patel/practicals/blob/bigdata/GraphX_Recommndation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.9 MB/s eta 0:00:00


In [3]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
from pyspark.sql.functions import col, explode

In [4]:
spark = SparkSession.builder \
    .appName("GraphFramesRecommendation") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.2-s_2.12") \
    .getOrCreate()

In [5]:
# 3️⃣ Create sample vertices (users + products)
vertices = spark.createDataFrame([
    ("u1", "user"),
    ("u2", "user"),
    ("u3", "user"),
    ("p1", "product"),
    ("p2", "product"),
    ("p3", "product"),
    ("p4", "product")
], ["id", "type"])

# 4️⃣ Create edges (purchase relationships)
edges = spark.createDataFrame([
    ("u1", "p1", "purchased"),
    ("u1", "p2", "purchased"),
    ("u2", "p2", "purchased"),
    ("u2", "p3", "purchased"),
    ("u3", "p3", "purchased"),
    ("u3", "p4", "purchased")
], ["src", "dst", "relationship"])

# 5️⃣ Build GraphFrame
g = GraphFrame(vertices, edges)

# 6️⃣ Find recommendations:
# Users who bought the same product as others → recommend their other products
paths = g.find("(u1)-[e1]->(p); (u2)-[e2]->(p); (u2)-[e3]->(otherP)") \
          .filter("u1.id != u2.id") \
          .select("u1.id", "otherP.id") \
          .distinct()

# 7️⃣ Show recommendations
paths.show()

/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+---+
| id| id|
+---+---+
| u1| p2|
| u3| p2|
| u2| p2|
| u2| p1|
| u1| p3|
| u3| p3|
| u2| p3|
| u2| p4|
+---+---+



Vertices: u1, u2, u3 (users) + p1, p2, p3, p4 (products).

Edges: (user → product) meaning the user purchased the product.

Motif Query (find):

(u1)-[e1]->(p) = User1 bought Product p

(u2)-[e2]->(p) = Another user also bought same Product p

(u2)-[e3]->(otherP) = That second user also bought another Product

Filter: u1.id != u2.id (exclude self).

Result: Recommend otherP to u1.